# 从零实现 Soft Actor-Critic：双 Q、熵温度与连续控制

本 Notebook 不调用 TorchRL、Stable-Baselines 或环境框架，从一个一维线性控制环境开始，手写 squashed Gaussian actor、双 Q critic、target network、replay buffer、自动 entropy temperature、Bellman target、软更新、训练与发布服务。

合成环境只验证 SAC 的数值与状态合同。回报改善不能外推到机器人控制；真实系统还需要仿真到现实差异、安全约束、控制频率、观测延迟、动作执行器和多随机种子评估。

In [ ]:
import copy, hashlib, io, json, math, random, warnings  # 导入本单元所需的依赖。
from collections import deque  # 导入本单元所需的依赖。
from dataclasses import dataclass  # 导入本单元所需的依赖。
from types import MappingProxyType  # 导入本单元所需的依赖。
warnings.filterwarnings("ignore", message="The pynvml package is deprecated")  # 计算并保存当前步骤的中间状态。
import numpy as np  # 导入本单元所需的依赖。
import torch  # 导入本单元所需的依赖。
from torch import nn  # 导入本单元所需的依赖。
import torch.nn.functional as F  # 导入本单元所需的依赖。

SEED64=6401  # 计算并保存当前步骤的中间状态。
random.seed(SEED64); np.random.seed(SEED64); torch.manual_seed(SEED64); torch.set_num_threads(1)  # 执行当前语句以推进本节示例。
DEVICE64=torch.device("cpu")  # 计算并保存当前步骤的中间状态。
def canonical64(x): return json.dumps(x,ensure_ascii=False,sort_keys=True,separators=(",",":"))  # 定义本节可复用的核心函数。
def sha64(x): return hashlib.sha256(x).hexdigest()  # 定义本节可复用的核心函数。
assert DEVICE64.type=="cpu" and torch.get_num_threads()==1  # 用受控断言验证关键不变量。

## 1. 连续环境与终止合同

状态、动作均为标量。动力学为 $s_{t+1}=0.8s_t+0.6a_t$，动作限定 `[-1,1]`，奖励为 $-(s_{t+1}^2+0.05a_t^2)$。完成固定 horizon 是 `terminated`；较短服务预算才是 `truncated`。训练 target 对 terminated 不 bootstrap，但 time-limit truncation 可以 bootstrap。

环境使用显式 generator 采样初态，禁止 NaN、越界动作和 done 后继续 step。

In [ ]:
class LinearControlEnv64:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,horizon=8,max_steps=None):  # 定义本节可复用的核心函数。
        self.horizon=int(horizon); self.max_steps=self.horizon if max_steps is None else int(max_steps)  # 计算并保存当前步骤的中间状态。
        if not 1<=self.max_steps<=self.horizon: raise ValueError("invalid_horizon")  # 按当前条件选择后续控制路径。
        self.state=None; self.step_count=0; self.done=True  # 计算并保存当前步骤的中间状态。
    def reset(self,generator):  # 定义本节可复用的核心函数。
        if not isinstance(generator,torch.Generator): raise TypeError("explicit_generator_required")  # 按当前条件选择后续控制路径。
        self.state=(torch.rand(1,generator=generator)*2-1)*1.8; self.step_count=0; self.done=False  # 计算并保存当前步骤的中间状态。
        return self.state.clone()  # 返回当前分支计算出的结果。
    def step(self,action):  # 定义本节可复用的核心函数。
        action=torch.as_tensor(action,dtype=torch.float32).reshape(-1)  # 计算并保存当前步骤的中间状态。
        if self.done: raise RuntimeError("episode_finished")  # 按当前条件选择后续控制路径。
        if action.shape!=(1,) or not torch.isfinite(action).all() or bool((action.abs()>1).any()): raise ValueError("action_contract")  # 按当前条件选择后续控制路径。
        next_state=0.8*self.state+0.6*action; reward=-(next_state.square()+0.05*action.square()).squeeze(0)  # 计算并保存当前步骤的中间状态。
        self.state=next_state; self.step_count+=1  # 计算并保存当前步骤的中间状态。
        terminated=self.step_count==self.horizon; truncated=self.step_count==self.max_steps and not terminated; self.done=terminated or truncated  # 计算并保存当前步骤的中间状态。
        return next_state.clone(),float(reward),terminated,truncated  # 返回当前分支计算出的结果。

env_probe64=LinearControlEnv64(); s64=env_probe64.reset(torch.Generator().manual_seed(2))  # 计算并保存当前步骤的中间状态。
ns64,r64,term64,trunc64=env_probe64.step(torch.tensor([0.0]))  # 计算并保存当前步骤的中间状态。
assert torch.allclose(ns64,0.8*s64) and r64<=0 and not term64 and not trunc64  # 用受控断言验证关键不变量。
short_env64=LinearControlEnv64(max_steps=1); short_env64.reset(torch.Generator().manual_seed(2)); _,_,t64,tr64=short_env64.step([0.])  # 计算并保存当前步骤的中间状态。
assert tr64 and not t64  # 用受控断言验证关键不变量。
try: env_probe64.step([1.1]); raise AssertionError("out-of-range action accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="action_contract"  # 捕获预期异常并验证失败分支。

## 2. Tanh-squashed Gaussian actor

actor 输出 `mean,log_std:[B,A]`，先重参数化 $u=\mu+\sigma\epsilon$，再令 $a=\tanh(u)$。变换后的 log-prob 必须减 Jacobian：
$$\log\pi(a|s)=\log\mathcal N(u;\mu,\sigma)-\sum_i\log(1-\tanh^2(u_i)+\epsilon).$$

漏掉修正会使 entropy 和温度学习错误。`log_std` 被限制在 `[-5,1]`，输出必须有限且动作严格位于 `[-1,1]`。

In [ ]:
class SquashedGaussianActor64(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,state_dim=1,action_dim=1,hidden=32):  # 定义本节可复用的核心函数。
        super().__init__(); self.state_dim=state_dim; self.action_dim=action_dim  # 计算并保存当前步骤的中间状态。
        self.body=nn.Sequential(nn.Linear(state_dim,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU())  # 计算并保存当前步骤的中间状态。
        self.mean=nn.Linear(hidden,action_dim); self.log_std=nn.Linear(hidden,action_dim)  # 计算并保存当前步骤的中间状态。
    def forward(self,state):  # 定义本节可复用的核心函数。
        if state.ndim!=2 or state.shape[1]!=self.state_dim or not torch.isfinite(state).all(): raise ValueError("actor_state_contract")  # 按当前条件选择后续控制路径。
        h=self.body(state); return self.mean(h),self.log_std(h).clamp(-5,1)  # 计算并保存当前步骤的中间状态。
    def sample(self,state,generator,deterministic=False):  # 定义本节可复用的核心函数。
        if not isinstance(generator,torch.Generator): raise TypeError("explicit_generator_required")  # 按当前条件选择后续控制路径。
        mean,log_std=self(state); noise=torch.zeros_like(mean) if deterministic else torch.randn(mean.shape,generator=generator,dtype=mean.dtype)  # 计算并保存当前步骤的中间状态。
        pre=mean+log_std.exp()*noise; action=torch.tanh(pre)  # 计算并保存当前步骤的中间状态。
        normal=-0.5*(((pre-mean)/log_std.exp()).square()+2*log_std+math.log(2*math.pi))  # 计算并保存当前步骤的中间状态。
        log_prob=(normal-torch.log1p(-action.square()+1e-6)).sum(-1)  # 计算并保存当前步骤的中间状态。
        if not torch.isfinite(action).all() or not torch.isfinite(log_prob).all(): raise ValueError("nonfinite_actor_output")  # 按当前条件选择后续控制路径。
        return action,log_prob,torch.tanh(mean)  # 返回当前分支计算出的结果。

actor_probe64=SquashedGaussianActor64(); state_probe64=torch.tensor([[0.2],[-0.4]])  # 计算并保存当前步骤的中间状态。
a64,lp64,m64=actor_probe64.sample(state_probe64,torch.Generator().manual_seed(3))  # 计算并保存当前步骤的中间状态。
assert a64.shape==(2,1) and lp64.shape==(2,) and bool((a64.abs()<=1).all())  # 用受控断言验证关键不变量。
mean64,ls64=actor_probe64(state_probe64); pre64=torch.atanh(a64.clamp(-.999999,.999999))  # 计算并保存当前步骤的中间状态。
manual_lp64=(-0.5*(((pre64-mean64)/ls64.exp()).square()+2*ls64+math.log(2*math.pi))-torch.log1p(-a64.square()+1e-6)).sum(-1)  # 计算并保存当前步骤的中间状态。
assert torch.allclose(lp64,manual_lp64,atol=2e-5)  # 用受控断言验证关键不变量。

## 3. 双 Q 与 target

两个独立 critic 接收 `[state,action]`，target 使用较小值降低正偏差：
$$y=r+\gamma(1-d)\left[\min(Q'_1,Q'_2)-\alpha\log\pi(a'|s')\right].$$

`d` 只表示真正 terminated；若任务选择不对 truncation bootstrap，必须作为发布超参另行编码。target 张量必须 detach，terminal 样本严格等于 reward。

In [ ]:
class QNetwork64(nn.Module):  # 定义承载本节状态与行为的数据结构。
    def __init__(self,state_dim=1,action_dim=1,hidden=32):  # 定义本节可复用的核心函数。
        super().__init__(); self.state_dim=state_dim; self.action_dim=action_dim  # 计算并保存当前步骤的中间状态。
        self.net=nn.Sequential(nn.Linear(state_dim+action_dim,hidden),nn.ReLU(),nn.Linear(hidden,hidden),nn.ReLU(),nn.Linear(hidden,1))  # 计算并保存当前步骤的中间状态。
    def forward(self,state,action):  # 定义本节可复用的核心函数。
        if state.ndim!=2 or action.ndim!=2 or state.shape[0]!=action.shape[0] or state.shape[1]!=self.state_dim or action.shape[1]!=self.action_dim: raise ValueError("q_shape_contract")  # 按当前条件选择后续控制路径。
        if not torch.isfinite(state).all() or not torch.isfinite(action).all(): raise ValueError("q_finite_contract")  # 按当前条件选择后续控制路径。
        return self.net(torch.cat([state,action],-1)).squeeze(-1)  # 返回当前分支计算出的结果。

def sac_target64(reward,terminated,next_q1,next_q2,next_logp,alpha,gamma=.97):  # 定义本节可复用的核心函数。
    if not reward.is_floating_point() or terminated.dtype!=torch.bool or any(x.shape!=reward.shape for x in (terminated,next_q1,next_q2,next_logp)): raise ValueError("target_contract")  # 按当前条件选择后续控制路径。
    if not math.isfinite(alpha) or alpha<=0 or not 0<=gamma<=1: raise ValueError("target_hyperparameter_contract")  # 按当前条件选择后续控制路径。
    target=reward+gamma*(~terminated).to(reward.dtype)*(torch.minimum(next_q1,next_q2)-alpha*next_logp)  # 计算并保存当前步骤的中间状态。
    if not torch.isfinite(target).all(): raise ValueError("nonfinite_target")  # 按当前条件选择后续控制路径。
    return target.detach()  # 返回当前分支计算出的结果。

target64=sac_target64(torch.tensor([1.,1.]),torch.tensor([True,False]),torch.tensor([9.,4.]),torch.tensor([8.,6.]),torch.tensor([2.,1.]),.2)  # 计算并保存当前步骤的中间状态。
assert torch.isclose(target64[0],torch.tensor(1.)) and torch.isclose(target64[1],torch.tensor(1.+.97*3.8))  # 用受控断言验证关键不变量。
assert not target64.requires_grad  # 用受控断言验证关键不变量。
q_probe64=QNetwork64(); assert q_probe64(state_probe64,a64).shape==(2,)  # 计算并保存当前步骤的中间状态。

## 4. Replay buffer 与数据质量

transition 保存 `state/action/reward/next_state/terminated/truncated`。buffer 在写入时验证 shape、范围、dtype 和 finite，并复制数据防止调用方后续原地修改。采样使用显式 generator。

在线 SAC 的数据分布由正在变化的策略产生；发布时要绑定环境版本、reward、action scale、replay warmup 和更新比率。

In [ ]:
@dataclass(frozen=True)  # 为下方定义附加声明式配置。
class Transition64:  # 定义承载本节状态与行为的数据结构。
    state: torch.Tensor; action: torch.Tensor; reward: float; next_state: torch.Tensor; terminated: bool; truncated: bool  # 执行当前语句以推进本节示例。

class ReplayBuffer64:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,capacity=5000): self.data=deque(maxlen=int(capacity))  # 定义本节可复用的核心函数。
    def add(self,t):  # 定义本节可复用的核心函数。
        if not isinstance(t,Transition64) or t.state.shape!=(1,) or t.next_state.shape!=(1,) or t.action.shape!=(1,): raise ValueError("transition_shape_contract")  # 按当前条件选择后续控制路径。
        if any(x.dtype!=torch.float32 for x in (t.state,t.action,t.next_state)): raise ValueError("transition_dtype_contract")  # 按当前条件选择后续控制路径。
        if not torch.isfinite(t.state).all() or not torch.isfinite(t.next_state).all() or not torch.isfinite(t.action).all() or not math.isfinite(t.reward): raise ValueError("transition_finite_contract")  # 按当前条件选择后续控制路径。
        if bool((t.action.abs()>1).any()) or not isinstance(t.terminated,bool) or not isinstance(t.truncated,bool) or (t.terminated and t.truncated): raise ValueError("transition_semantic_contract")  # 按当前条件选择后续控制路径。
        self.data.append(Transition64(t.state.clone(),t.action.clone(),float(t.reward),t.next_state.clone(),t.terminated,t.truncated))  # 执行当前语句以推进本节示例。
    def sample(self,batch_size,generator):  # 定义本节可复用的核心函数。
        if len(self.data)<batch_size or not isinstance(generator,torch.Generator): raise ValueError("replay_sample_contract")  # 按当前条件选择后续控制路径。
        idx=torch.randint(0,len(self.data),(batch_size,),generator=generator); rows=[self.data[int(i)] for i in idx]  # 计算并保存当前步骤的中间状态。
        return {"state":torch.stack([r.state for r in rows]),"action":torch.stack([r.action for r in rows]),"reward":torch.tensor([r.reward for r in rows]),"next_state":torch.stack([r.next_state for r in rows]),"terminated":torch.tensor([r.terminated for r in rows]),"truncated":torch.tensor([r.truncated for r in rows])}  # 返回当前分支计算出的结果。
    def __len__(self): return len(self.data)  # 定义本节可复用的核心函数。

replay_probe64=ReplayBuffer64(); replay_probe64.add(Transition64(torch.tensor([.2]),torch.tensor([0.]),-0.1,torch.tensor([.1]),False,False))  # 计算并保存当前步骤的中间状态。
copied64=replay_probe64.data[0].state.clone();  # 计算并保存当前步骤的中间状态。
try: replay_probe64.add(Transition64(torch.tensor([float("nan")]),torch.tensor([0.]),0.,torch.tensor([0.]),False,False)); raise AssertionError("NaN replay accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="transition_finite_contract"  # 捕获预期异常并验证失败分支。
try: replay_probe64.add(Transition64(torch.tensor([1]),torch.tensor([0.]),0.,torch.tensor([0.]),False,False)); raise AssertionError("integer replay accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="transition_dtype_contract"  # 捕获预期异常并验证失败分支。
assert torch.equal(copied64,replay_probe64.data[0].state)  # 用受控断言验证关键不变量。

## 5. SAC update、温度与 Polyak 平均

每次更新先优化双 Q，再冻结 Q 参数计算 actor loss $E[\alpha\log\pi-\min Q]$，最后优化 `log_alpha` 使 entropy 接近 `target_entropy=-action_dim`。target 参数用 $\theta'\leftarrow(1-\tau)\theta'+\tau\theta$。

冻结 Q 只阻止无用 critic 梯度，动作到 Q 的梯度仍会回到 actor。alpha 用 log 参数保证为正。

In [ ]:
class SACAgent64:  # 定义承载本节状态与行为的数据结构。
    def __init__(self):  # 定义本节可复用的核心函数。
        self.actor=SquashedGaussianActor64(); self.q1=QNetwork64(); self.q2=QNetwork64(); self.tq1=copy.deepcopy(self.q1); self.tq2=copy.deepcopy(self.q2)  # 计算并保存当前步骤的中间状态。
        self.tq1.requires_grad_(False); self.tq2.requires_grad_(False)  # 执行当前语句以推进本节示例。
        self.actor_opt=torch.optim.Adam(self.actor.parameters(),lr=3e-3); self.q_opt=torch.optim.Adam(list(self.q1.parameters())+list(self.q2.parameters()),lr=3e-3)  # 计算并保存当前步骤的中间状态。
        self.log_alpha=torch.tensor(math.log(.2),requires_grad=True); self.alpha_opt=torch.optim.Adam([self.log_alpha],lr=1e-3); self.target_entropy=-1.0  # 计算并保存当前步骤的中间状态。
    @property  # 为下方定义附加声明式配置。
    def alpha(self): return self.log_alpha.exp()  # 定义本节可复用的核心函数。
    def update(self,batch,generator,gamma=.97,tau=.02):  # 定义本节可复用的核心函数。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            next_action,next_logp,_=self.actor.sample(batch["next_state"],generator); nq1=self.tq1(batch["next_state"],next_action); nq2=self.tq2(batch["next_state"],next_action)  # 计算并保存当前步骤的中间状态。
            y=sac_target64(batch["reward"],batch["terminated"],nq1,nq2,next_logp,float(self.alpha),gamma)  # 计算并保存当前步骤的中间状态。
        q1=self.q1(batch["state"],batch["action"]); q2=self.q2(batch["state"],batch["action"]); q_loss=F.mse_loss(q1,y)+F.mse_loss(q2,y)  # 计算并保存当前步骤的中间状态。
        self.q_opt.zero_grad(set_to_none=True); q_loss.backward(); self.q_opt.step()  # 计算并保存当前步骤的中间状态。
        for p in list(self.q1.parameters())+list(self.q2.parameters()): p.requires_grad_(False)  # 遍历输入元素以累积或检查结果。
        action,logp,_=self.actor.sample(batch["state"],generator); actor_loss=(self.alpha.detach()*logp-torch.minimum(self.q1(batch["state"],action),self.q2(batch["state"],action))).mean()  # 计算并保存当前步骤的中间状态。
        self.actor_opt.zero_grad(set_to_none=True); actor_loss.backward(); self.actor_opt.step()  # 计算并保存当前步骤的中间状态。
        for p in list(self.q1.parameters())+list(self.q2.parameters()): p.requires_grad_(True)  # 遍历输入元素以累积或检查结果。
        alpha_loss=-(self.log_alpha*(logp.detach()+self.target_entropy)).mean(); self.alpha_opt.zero_grad(set_to_none=True); alpha_loss.backward(); self.alpha_opt.step()  # 计算并保存当前步骤的中间状态。
        with torch.no_grad():  # 在受管理的上下文中执行操作。
            for target,source in ((self.tq1,self.q1),(self.tq2,self.q2)):  # 遍历输入元素以累积或检查结果。
                for tp,sp in zip(target.parameters(),source.parameters()): tp.lerp_(sp,tau)  # 遍历输入元素以累积或检查结果。
        return float(q_loss),float(actor_loss),float(self.alpha)  # 返回当前分支计算出的结果。

agent_probe64=SACAgent64()  # 计算并保存当前步骤的中间状态。
before_target64=[p.clone() for p in agent_probe64.tq1.parameters()]  # 计算并保存当前步骤的中间状态。
assert agent_probe64.alpha>0 and all(not p.requires_grad for p in agent_probe64.tq1.parameters())  # 用受控断言验证关键不变量。
assert all(torch.allclose(a,b) for a,b in zip(agent_probe64.q1.parameters(),agent_probe64.tq1.parameters()))  # 用受控断言验证关键不变量。

### 5.1 单步更新的梯度与 target oracle

训练曲线变好并不能证明更新顺序正确。这里额外构造一个完全可控的 batch，逐项验证四条工程不变量：online critic 确实更新；target 只通过 Polyak 平均移动且始终不参与反向传播；actor 与 `log_alpha` 都收到有限梯度；terminal mask 与超参错误能够在进入优化器前暴露。

这些 oracle 很适合写进真实项目的单元测试。它们比“loss 能下降”更容易定位 stop-gradient、参数冻结、bootstrap 和随机数复现问题。

In [ ]:
oracle_batch64={  # 计算并保存当前步骤的中间状态。
    "state":torch.tensor([[-1.],[-.2],[.3],[1.]]),  # 执行当前语句以推进本节示例。
    "action":torch.tensor([[.5],[.1],[-.2],[-.6]]),  # 执行当前语句以推进本节示例。
    "reward":torch.tensor([-.2,-.1,-.1,-.3]),  # 执行当前语句以推进本节示例。
    "next_state":torch.tensor([[-.5],[-.1],[.1],[.4]]),  # 执行当前语句以推进本节示例。
    "terminated":torch.tensor([False,True,False,True]),  # 执行当前语句以推进本节示例。
    "truncated":torch.tensor([False,False,True,False]),  # 执行当前语句以推进本节示例。
}  # 执行当前语句以推进本节示例。
before_q64=[p.detach().clone() for p in agent_probe64.q1.parameters()]  # 计算并保存当前步骤的中间状态。
before_t64=[p.detach().clone() for p in agent_probe64.tq1.parameters()]  # 计算并保存当前步骤的中间状态。
before_alpha64=float(agent_probe64.alpha)  # 计算并保存当前步骤的中间状态。
metrics_probe64=agent_probe64.update(oracle_batch64,torch.Generator().manual_seed(64),tau=.5)  # 计算并保存当前步骤的中间状态。
assert len(metrics_probe64)==3 and all(math.isfinite(x) for x in metrics_probe64)  # 用受控断言验证关键不变量。
assert any(not torch.equal(a,b) for a,b in zip(before_q64,agent_probe64.q1.parameters()))  # 用受控断言验证关键不变量。
assert any(not torch.equal(a,b) for a,b in zip(before_t64,agent_probe64.tq1.parameters()))  # 用受控断言验证关键不变量。
assert all(not p.requires_grad and p.grad is None for p in agent_probe64.tq1.parameters())  # 用受控断言验证关键不变量。
assert all(p.requires_grad for p in agent_probe64.q1.parameters())  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in agent_probe64.actor.parameters())  # 用受控断言验证关键不变量。
assert agent_probe64.log_alpha.grad is not None and torch.isfinite(agent_probe64.log_alpha.grad)  # 用受控断言验证关键不变量。
assert float(agent_probe64.alpha)>0 and float(agent_probe64.alpha)!=before_alpha64  # 用受控断言验证关键不变量。
boot64=sac_target64(torch.tensor([1.]),torch.tensor([False]),torch.tensor([2.]),torch.tensor([3.]),torch.tensor([0.]),.2)  # 计算并保存当前步骤的中间状态。
assert boot64.item()>1.  # 用受控断言验证关键不变量。
try: sac_target64(torch.tensor([1.]),torch.tensor([False]),torch.tensor([2.]),torch.tensor([3.]),torch.tensor([0.]),.2,1.1); raise AssertionError("invalid gamma accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="target_hyperparameter_contract"  # 捕获预期异常并验证失败分支。
try: actor_probe64.sample(state_probe64,None); raise AssertionError("implicit randomness accepted")  # 尝试执行可能失败的受控操作。
except TypeError as e: assert str(e)=="explicit_generator_required"  # 捕获预期异常并验证失败分支。

## 6. 在线受控训练

前 96 步随机探索，之后从 actor 采样；replay 达到 64 条后每环境步更新一次。训练和评估 generator 分离。评估固定一组初态并使用 deterministic actor，比较未训练与训练后的平均回报。

这是低维短 horizon smoke test，不代表复杂动力学的样本效率。target、temperature、reward scale 和 update-to-data ratio 都需要单独监控。

In [ ]:
def evaluate64(agent,initial_states):  # 定义本节可复用的核心函数。
    returns=[]  # 返回当前分支计算出的结果。
    for initial in initial_states:  # 遍历输入元素以累积或检查结果。
        env=LinearControlEnv64(); env.state=torch.tensor([initial]); env.step_count=0; env.done=False; total=0.  # 计算并保存当前步骤的中间状态。
        while not env.done:  # 在终止条件满足前持续推进状态。
            with torch.no_grad(): action=agent.actor.sample(env.state[None,:],torch.Generator().manual_seed(0),True)[2].squeeze(0)  # 在受管理的上下文中执行操作。
            _,reward,_,_=env.step(action); total+=reward  # 计算并保存当前步骤的中间状态。
        returns.append(total)  # 返回当前分支计算出的结果。
    return float(np.mean(returns))  # 返回当前分支计算出的结果。

torch.manual_seed(SEED64); agent64=SACAgent64(); replay64=ReplayBuffer64(); env64=LinearControlEnv64()  # 计算并保存当前步骤的中间状态。
env_generator64=torch.Generator().manual_seed(SEED64+1); action_generator64=torch.Generator().manual_seed(SEED64+2); sample_generator64=torch.Generator().manual_seed(SEED64+3)  # 计算并保存当前步骤的中间状态。
eval_starts64=[-1.6,-.8,.8,1.6]; initial_return64=evaluate64(agent64,eval_starts64)  # 计算并保存当前步骤的中间状态。
state64=env64.reset(env_generator64); logs64=[]  # 计算并保存当前步骤的中间状态。
for step64 in range(1150):  # 遍历输入元素以累积或检查结果。
    if step64<96: action64=torch.rand(1,generator=action_generator64)*2-1  # 按当前条件选择后续控制路径。
    else:  # 处理前置条件不成立的分支。
        with torch.no_grad(): action64=agent64.actor.sample(state64[None,:],action_generator64)[0].squeeze(0)  # 在受管理的上下文中执行操作。
    next_state64,reward64,terminated64,truncated64=env64.step(action64)  # 计算并保存当前步骤的中间状态。
    replay64.add(Transition64(state64,action64,reward64,next_state64,terminated64,truncated64)); state64=next_state64  # 计算并保存当前步骤的中间状态。
    if terminated64 or truncated64: state64=env64.reset(env_generator64)  # 按当前条件选择后续控制路径。
    if len(replay64)>=64: logs64.append(agent64.update(replay64.sample(64,sample_generator64),action_generator64))  # 按当前条件选择后续控制路径。
final_return64=evaluate64(agent64,eval_starts64)  # 计算并保存当前步骤的中间状态。
assert final_return64>initial_return64+0.4  # 用受控断言验证关键不变量。
assert all(math.isfinite(v) for row in logs64[-20:] for v in row) and 0<float(agent64.alpha)<2  # 用受控断言验证关键不变量。
assert any(p.grad is not None and torch.isfinite(p.grad).all() for p in agent64.actor.parameters())  # 用受控断言验证关键不变量。
print({"initial_return":round(initial_return64,4),"final_return":round(final_return64,4),"alpha":round(float(agent64.alpha),4)})  # 执行当前语句以推进本节示例。

## 7. 发布服务与制品边界

发布 manifest 绑定动力学、reward、state/action scale、actor config、训练步数、warmup、update ratio、gamma/tau、temperature 和评估初态。state 摘要逐 tensor 覆盖 key/dtype/shape/bytes；包外只读 registry 是信任锚。

`PublishedSAC64` 只接受命名状态 `{"position":...}`，内部检查 finite/range 并返回带动作名和范围的结果，避免调用方错用尺度。

In [ ]:
def tensor_hash64(t):  # 定义本节可复用的核心函数。
    v=t.detach().cpu().contiguous(); return sha64(str(v.dtype).encode()+canonical64(list(v.shape)).encode()+v.numpy().tobytes())  # 计算并保存当前步骤的中间状态。
def state_digest64(state):  # 定义本节可复用的核心函数。
    h=hashlib.sha256()  # 计算并保存当前步骤的中间状态。
    for k,v in sorted(state.items()): h.update(k.encode()); h.update(tensor_hash64(v).encode())  # 遍历输入元素以累积或检查结果。
    return h.hexdigest()  # 返回当前分支计算出的结果。
manifest64={"artifact_id":"sac-linear-control-v1","version":1,"actor_config":{"state_dim":1,"action_dim":1,"hidden":32},"observation":{"position_range":[-2.,2.]},"action":{"name":"control","range":[-1.,1.]},"environment":{"transition":"0.8*s+0.6*a","reward":"-(next_state^2+0.05*a^2)","horizon":8,"truncation_bootstrap":True},"replay":{"capacity":5000,"transition_dtype":"float32","sampling":"uniform_with_replacement"},"training":{"seed":SEED64,"steps":1150,"warmup":96,"batch":64,"update_to_data_ratio_after_warmup":1.0,"gamma":.97,"tau":.02,"actor_lr":.003,"critic_lr":.003,"alpha_lr":.001,"initial_alpha":.2,"target_entropy":-1.,"target_update":"after_actor"},"evaluation":{"initial_states":eval_starts64}}  # 计算并保存当前步骤的中间状态。
def build_package64(model,manifest):  # 定义本节可复用的核心函数。
    b=io.BytesIO(); torch.save(model.state_dict(),b); raw=b.getvalue(); state=torch.load(io.BytesIO(raw),map_location="cpu",weights_only=True)  # 计算并保存当前步骤的中间状态。
    ms=sha64(canonical64(manifest).encode()); sd=state_digest64(state); rs=sha64(raw); bundle=sha64(canonical64({"m":ms,"s":sd,"r":rs}).encode())  # 计算并保存当前步骤的中间状态。
    return {"manifest":copy.deepcopy(manifest),"manifest_sha":ms,"state_bytes":raw,"state_digest":sd,"state_bytes_sha":rs,"bundle_digest":bundle}  # 返回当前分支计算出的结果。
package64=build_package64(agent64.actor,manifest64); REGISTRY64=MappingProxyType({("sac-linear-control-v1",1):package64["bundle_digest"]})  # 计算并保存当前步骤的中间状态。
class PublishedSAC64:  # 定义承载本节状态与行为的数据结构。
    def __init__(self,actor,manifest):  # 定义本节可复用的核心函数。
        self._actor=actor  # 计算并保存当前步骤的中间状态。
        self.observation=MappingProxyType({"position_range":tuple(manifest["observation"]["position_range"])})  # 计算并保存当前步骤的中间状态。
        self.action=MappingProxyType({"name":manifest["action"]["name"],"range":tuple(manifest["action"]["range"])})  # 计算并保存当前步骤的中间状态。
    @torch.no_grad()  # 为下方定义附加声明式配置。
    def act(self,named_state):  # 定义本节可复用的核心函数。
        if set(named_state)!={"position"}: raise ValueError("named_state_schema")  # 按当前条件选择后续控制路径。
        x=torch.as_tensor(named_state["position"],dtype=torch.float32).reshape(-1)  # 计算并保存当前步骤的中间状态。
        low,high=self.observation["position_range"]  # 计算并保存当前步骤的中间状态。
        if x.numel()==0 or not torch.isfinite(x).all() or bool(((x<low)|(x>high)).any()): raise ValueError("position_contract")  # 按当前条件选择后续控制路径。
        action=self._actor.sample(x[:,None],torch.Generator().manual_seed(0),True)[2].squeeze(1)  # 计算并保存当前步骤的中间状态。
        return MappingProxyType({"control":action.clone(),"range":self.action["range"]})  # 返回当前分支计算出的结果。
def load64(package):  # 定义本节可复用的核心函数。
    m=package["manifest"]; key=(m.get("artifact_id"),m.get("version"))  # 计算并保存当前步骤的中间状态。
    current_ms=sha64(canonical64(m).encode()); current_rs=sha64(package["state_bytes"])  # 计算并保存当前步骤的中间状态。
    if m!=manifest64 or current_ms!=package["manifest_sha"] or current_rs!=package["state_bytes_sha"]: raise RuntimeError("package_contract")  # 按当前条件选择后续控制路径。
    state=torch.load(io.BytesIO(package["state_bytes"]),map_location="cpu",weights_only=True)  # 计算并保存当前步骤的中间状态。
    current_sd=state_digest64(state)  # 计算并保存当前步骤的中间状态。
    if current_sd!=package["state_digest"]: raise RuntimeError("state_contract")  # 按当前条件选择后续控制路径。
    current_bundle=sha64(canonical64({"m":current_ms,"s":current_sd,"r":current_rs}).encode())  # 计算并保存当前步骤的中间状态。
    if package.get("bundle_digest")!=current_bundle: raise RuntimeError("bundle_contract")  # 按当前条件选择后续控制路径。
    if REGISTRY64.get(key)!=current_bundle: raise RuntimeError("publisher_registry_rejected")  # 按当前条件选择后续控制路径。
    actor=SquashedGaussianActor64(**m["actor_config"]); actor.load_state_dict(state); actor.eval(); return PublishedSAC64(actor,m)  # 计算并保存当前步骤的中间状态。
published64=load64(package64); served64=published64.act({"position":[-.5,.5]})  # 计算并保存当前步骤的中间状态。
assert served64["control"].shape==(2,) and bool((served64["control"].abs()<=1).all())  # 用受控断言验证关键不变量。
assert torch.equal(served64["control"],published64.act({"position":[-.5,.5]})["control"])  # 用受控断言验证关键不变量。
assert isinstance(published64.observation,MappingProxyType) and isinstance(served64["range"],tuple)  # 用受控断言验证关键不变量。
try: published64.act({"state":[0.]}); raise AssertionError("unnamed position accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="named_state_schema"  # 捕获预期异常并验证失败分支。
try: published64.act({"position":[float("inf")]}); raise AssertionError("nonfinite position accepted")  # 尝试执行可能失败的受控操作。
except ValueError as e: assert str(e)=="position_contract"  # 捕获预期异常并验证失败分支。
forged64=build_package64(SquashedGaussianActor64(),manifest64)  # 计算并保存当前步骤的中间状态。
try: load64(forged64); raise AssertionError("re-signed actor accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="publisher_registry_rejected"  # 捕获预期异常并验证失败分支。
forged_old_bundle64=copy.deepcopy(forged64); forged_old_bundle64["bundle_digest"]=package64["bundle_digest"]  # 计算并保存当前步骤的中间状态。
try: load64(forged_old_bundle64); raise AssertionError("forged actor with old bundle accepted")  # 尝试执行可能失败的受控操作。
except RuntimeError as e: assert str(e)=="bundle_contract"  # 捕获预期异常并验证失败分支。

## 8. 失败模式、复杂度与来源

常见错误：漏 tanh Jacobian；用单 Q；target 未 detach；把 truncation 当 terminal；actor 更新时给 critic 留无用梯度；alpha 符号错误；replay 接受 NaN；服务改变动作尺度。每次 update 的主要成本是多次 actor/critic forward/backward，线上动作只需 actor 一次前向。

- Haarnoja et al., [Soft Actor-Critic](https://arxiv.org/abs/1801.01290), ICML 2018。
- Haarnoja et al., [SAC Algorithms and Applications](https://arxiv.org/abs/1812.05905), 2018。
- Fujimoto et al., [TD3](https://arxiv.org/abs/1802.09477)，双 critic 与 target smoothing 背景。